In [69]:
from pathlib import Path
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import pathlib
import pandas as pd
import re
from sklearn.model_selection import train_test_split
import torch.optim as optim
import time
import sys
#sys.path.append("/Users/deborahfu/eecs281/eecs545/Project/neurovfm")

#print(os.path.exists("/Users/deborahfu/eecs281/eecs545/Project/neurovfm"))

from mil import pad_ragged, AggregateThenClassify, ClassifyThenAggregate
     


In [49]:
folder_path = '/Users/deborahfu/eecs281/eecs545/Project/embeddings'
sum_ = 0.0
sq_sum = 0.0
count = 0
for f in os.listdir(folder_path):
    if f.endswith(".pt"):
        x = torch.load(os.path.join(folder_path, f)).float()
        sum_ += x.sum().item()
        sq_sum += (x**2).sum().item()
        count += x.numel()
mean = sum_ / count
std = ((sq_sum / count) - mean ** 2) ** 0.5
print("mean:", mean)
print("std:", std)

mean: -0.0034458619234182426
std: 0.05875309376332267


In [71]:
path = pathlib.Path(folder_path)
data_list = []
subject_meta = pd.read_csv('/Users/deborahfu/eecs281/eecs545/Project/for_training_participants.tsv', sep='\t')

filename_pattern = re.compile(r'sub-([^_]+)')
for file in path.glob('*.pt'):
    match = filename_pattern.search(file.name)
    subj_id = match.group(1)
    emb_tensor = torch.load(file, map_location='cpu')

    emb_tensor_np = emb_tensor.detach().cpu()
    data_list.append({
        'subject id': subj_id, 
        'embedding': emb_tensor_np
    })

emb_df = pd.DataFrame(data_list)
subject_meta['participant_id'] =  subject_meta['participant_id'].astype(str)
age_embedding_df = pd.merge(emb_df, subject_meta, left_on='subject id', right_on='participant_id', how='inner')

train_df, eval_df = train_test_split(
    emb_df,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

In [72]:
print(train_df.columns)

Index(['subject id', 'embedding'], dtype='object')


In [73]:
train_embedding_list = []
train_seq_pos = [0]
train_marker = 0
eval_embedding_list = []
eval_seq_pos = [0]
eval_marker = 0
emb_folder_path = "/Users/deborahfu/eecs281/eecs545/Project/embeddings"

for __, row in train_df.iterrows():
    path = os.path.join(emb_folder_path, "sub-" + row['subject id'] + "_encoder_embeddings.pt")
    data = torch.load(path)
    train_embedding_list.append(data)
    train_marker = train_marker+data.shape[0]
    train_seq_pos.append(train_marker)

for __, row in eval_df.iterrows():
    path = os.path.join(emb_folder_path, "sub-" + row['subject id'] + "_encoder_embeddings.pt")
    data = torch.load(path)
    eval_embedding_list.append(data)
    eval_marker = eval_marker+data.shape[0]
    eval_seq_pos.append(eval_marker)






In [78]:
all_train_data = torch.cat(train_embedding_list, dim = 0)
all_eval_data = torch.cat(eval_embedding_list, dim = 0)
#print(all_train_data.shape)
#print(train_seq_pos)

In [ ]:
train_seq_pos = torch.tensor(train_seq_pos, dtype = torch.long) #this dtype has to be long
train_rag_tensor = pad_ragged(all_train_data, train_seq_pos)

eval_seq_pos = torch.tensor(eval_seq_pos, dtype = torch.long)
eval_rag_tensor = pad_ragged(all_eval_data, eval_seq_pos)


/var/folders/33/lq8mg1px5zj3kgg0fx6vd3wm0000gn/T/ipykernel_24139/2219582644.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_seq_pos = torch.tensor(train_seq_pos, dtype = torch.long)


In [66]:
class ABMILRegression(nn.Module):
    def __init__(self, dim, hidden_dim=None):
        super().__init__()
        self.pooling = AggregateThenClassify(dim=dim, hidden_dim = hidden_dim, W_out=1)
        self.regression = nn.Linear(dim,1)
    def forward(self, emb_list, seq_position):
        bags = self.pool(emb_list, seq_position)
        output = self.regression(bags).squeeze(-1)
        return output

In [ ]:
#cell for inspecting embeddings, sanity check
data1 = torch.load('sub-105046731194_encoder_embeddings.pt')
data2 = torch.load('sub-105819777108_encoder_embeddings.pt')
data3 = torch.load('sub-106422294651_encoder_embeddings.pt')
data4 = torch.load('sub-107989352584_encoder_embeddings.pt')
data5 = torch.load('sub-108597937741_encoder_embeddings.pt')
print(data1.shape)
print(data2.shape)
print(data3.shape)
print(data4.shape)
print(data5.shape)

torch.Size([776, 768])
torch.Size([788, 768])
torch.Size([783, 768])
torch.Size([784, 768])
torch.Size([773, 768])


In [85]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ABMILRegression(dim = 768, hidden_dim = 256).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
print(f'Model initialize on {device}')

ImportError: FusedDense from flash_attn is required for MIL modules

In [ ]:
def train(model, train_concactenated, train_pos, train_ages,
          eval_concactenated, eval_pos, eval_ages, epochs=100):
    train_losses = []
    eval_losses = []
    
    model.train()
    train_loss = 0.0

    train_concactenated = train_concactenated.to(device, dtype=torch.float32)
    train_pos = train_pos.to(device, dtype = torch.float32)
    train_ages = train_ages.to(device, dtype = torch.float32)

    eval_concactenated = eval_concactenated.to(device, dtype=torch.float32)
    eval_pos = eval_pos.to(device, dtype = torch.float32)
    eval_ages = eval_ages.to(device, dtype = torch.float32)

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        train_outputs = model(train_concactenated, train_pos)
        train_loss = criterion(train_outputs, train_ages)

        train_loss.backward()
        optimizer.step()

        model.eval()
        
        with torch.no_grad():
            eval_outputs = model(eval_concactenated, eval_pos)
            eval_loss = criterion(eval_outputs, eval_ages)

        train_losses.append(train_loss.item())
        eval_losses.append(eval_loss.item())

        print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {train_loss.item():.4f} | Val Loss: {eval_loss.item():.4f}")
    
    return train_losses, eval_losses, eval_ages.detach().cpu(), eval_outputs.detach().cpu()

In [ ]:
start_time = time.time()
train_losses, val_losses, age, outputs = train(model, train_loader, val_loader)
end_time = time.time()
total_time = end_time - start_time
print(f" Training Complete!")
print(f"Total time: {int(total_time // 60)}m {int(total_time % 60)}s")